## Noise Input Lowres

In [2]:
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as T
from torchvision import models
from PIL import Image
import torch.nn.functional as F
import os

# --- SETUP ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).to(DEVICE).eval()

# Mapping VGG16 features to readable block names
BLOCK_MAP = {
    "Block1_Conv2": 2,
    "Block2_Conv2": 7,
    "Block3_Conv3": 14,
    "Block4_Conv3": 21,
    "Block5_Conv3": 28
}

# --- HELPERS ---

def laplacian_sharpen(img, alpha=0.5):
    """Reinforces edges by calculating the difference between image and its blurred version."""
    blurrer = T.GaussianBlur(kernel_size=3, sigma=1.0)
    return img + alpha * (img - blurrer(img))

def deprocess(img):
    """Converts a torch tensor to a displayable numpy array."""
    img = img.detach().cpu().squeeze(0).numpy().transpose(1, 2, 0)
    # Contrast stretching for extra visual punch
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)

def visualize_target_upscaled(target_type, layer_idx=None, filter_idx=None, class_idx=None, octaves=3, scale_factor=1.4):
    """Optimizes an image across multiple scales (octaves) to generate sharp 224x224 features."""
    # Start at a smaller resolution to establish global structure
    size = int(224 / (scale_factor ** (octaves - 1)))
    img = torch.randn(1, 3, size, size).to(DEVICE) * 0.05
    
    act_buffer = {}
    def hook_fn(m, i, o): act_buffer['val'] = o
    handle = None
    if target_type == 'filter':
        handle = model.features[layer_idx].register_forward_hook(hook_fn)

    for octave in range(octaves):
        img = img.detach().requires_grad_(True)
        optimizer = torch.optim.Adam([img], lr=0.05)
        
        # Optimize at current scale
        iterations = 60 if target_type == 'class' else 40
        for _ in range(iterations):
            optimizer.zero_grad()
            
            # Forward pass
            if target_type == 'class':
                output = model(img)
                loss = -output[0, class_idx]
            else:
                model(img)
                loss = -act_buffer['val'][0, filter_idx].mean()
            
            # Total Variation Regularization (reduces noise/pixelation)
            loss += 0.1 * (torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).mean() + 
                           torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).mean())
            loss.backward()
            optimizer.step()
        
        # Upscale to next resolution
        if octave < octaves - 1:
            new_size = int(size * scale_factor)
            img = F.interpolate(img, size=(new_size, new_size), mode='bilinear', align_corners=False)
            img = laplacian_sharpen(img, alpha=0.2) # Sharpen during transition
            size = new_size
            
    if handle: handle.remove()
    
    # Final resize to standard VGG 224x224 size
    img = F.interpolate(img, size=(224, 224), mode='bilinear', align_corners=False)
    return deprocess(img)

def get_top_filters(layer_idx, class_idx, num_filters=3):
    """Identifies which filters in a layer are most active for a specific class."""
    img = torch.randn(1, 3, 224, 224).to(DEVICE) * 0.05
    img.requires_grad_(True)
    opt = torch.optim.Adam([img], lr=0.1)
    
    # Quick optimization to get a class-relevant input
    for _ in range(15):
        opt.zero_grad()
        loss = -model(img)[0, class_idx]
        loss.backward()
        opt.step()
        
    activations = []
    def hook(m, i, o): activations.append(o.detach())
    handle = model.features[layer_idx].register_forward_hook(hook)
    model(img)
    handle.remove()
    
    filter_scores = activations[0][0].mean(dim=(1, 2))
    return torch.topk(filter_scores, num_filters).indices.cpu().numpy()

# --- MAIN EXECUTION ---

# Add ImageNet IDs here (e.g., 323: Monarch Butterfly, 130: Flamingo)
CLASSES_TO_VISUALIZE = {
    985: "Daisy_Floser",
}

OUTPUT_ROOT = "vgg_visualizations"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for class_idx, class_name in CLASSES_TO_VISUALIZE.items():
    print(f"\n>>> Starting visualizations for: {class_name}")
    
    # Create directory for the class
    class_dir = os.path.join(OUTPUT_ROOT, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    # 1. Generate Class Archetype (The "Ideal" image of that class)
    print(f"   Generating Class Archetype...")
    archetype = visualize_target_upscaled('class', class_idx=class_idx, octaves=4)
    Image.fromarray(archetype).save(os.path.join(class_dir, f"00_{class_name}_archetype.png"))
    
    # 2. Generate Filter Visualizations for each block
    for block_name, layer_idx in BLOCK_MAP.items():
        print(f"   Optimizing filters for {block_name}...")
        
        # Create subfolder for the layer
        layer_dir = os.path.join(class_dir, block_name)
        os.makedirs(layer_dir, exist_ok=True)
        
        # Find the top 3 most relevant filters for this class at this depth
        top_filters = get_top_filters(layer_idx, class_idx, num_filters=3)
        
        for i, f_idx in enumerate(top_filters):
            filter_img = visualize_target_upscaled('filter', layer_idx=layer_idx, filter_idx=f_idx, octaves=3)
            
            # Save individual 224x224 image
            file_name = f"rank{i+1}_filter{f_idx}.png"
            Image.fromarray(filter_img).save(os.path.join(layer_dir, file_name))

print(f"\nDone! All images are located in the '{OUTPUT_ROOT}' folder.")


>>> Starting visualizations for: Daisy_Floser
   Generating Class Archetype...
   Optimizing filters for Block1_Conv2...
   Optimizing filters for Block2_Conv2...
   Optimizing filters for Block3_Conv3...
   Optimizing filters for Block4_Conv3...
   Optimizing filters for Block5_Conv3...

Done! All images are located in the 'vgg_visualizations' folder.


## Noise Input Highres

In [4]:
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as T
from torchvision import models
from PIL import Image
import torch.nn.functional as F
import os

# --- INITIALIZATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).to(DEVICE).eval()

# VGG16 Layer mapping for structured exploration
BLOCK_MAP = {
    "Block1_Conv2": 2,
    "Block2_Conv2": 7,
    "Block3_Conv3": 14,
    "Block4_Conv3": 21,
    "Block5_Conv3": 28
}

# --- CORE UTILITIES ---

def laplacian_sharpen(img, alpha=0.4):
    """Enhances high-frequency edges to prevent blur during upscaling."""
    blur = T.GaussianBlur(kernel_size=3, sigma=1.0)(img)
    return img + alpha * (img - blur)

def deprocess(img):
    """Converts torch tensor to a clean 0-255 numpy array."""
    img = img.detach().cpu().squeeze(0).numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1) # Standardize range for natural colors
    return (img * 255).astype(np.uint8)

def get_top_filters(layer_idx, class_idx, num_filters=3):
    """Finds the specific neurons in a layer that respond most to the chosen class."""
    img = torch.randn(1, 3, 224, 224).to(DEVICE) * 0.05
    img.requires_grad_(True)
    opt = torch.optim.Adam([img], lr=0.1)
    
    # Quick optimization to find class-relevant activation patterns
    for _ in range(15):
        opt.zero_grad()
        loss = -model(img)[0, class_idx]
        loss.backward()
        opt.step()
        
    activations = []
    def hook(m, i, o): activations.append(o.detach())
    handle = model.features[layer_idx].register_forward_hook(hook)
    model(img)
    handle.remove()
    
    filter_scores = activations[0][0].mean(dim=(1, 2))
    return torch.topk(filter_scores, num_filters).indices.cpu().numpy()

# --- THE NATURAL SHARP ENGINE ---

def visualize_natural_sharp(target_type, layer_idx=None, filter_idx=None, class_idx=None, octaves=4, scale_factor=1.3):
    """Generates a 224x224 visualization using multi-scale optimization and color constraints."""
    
    # Start with neutral gray noise (calmer starting point for colors)
    size = int(224 / (scale_factor ** (octaves - 1)))
    img = torch.ones(1, 3, size, size).to(DEVICE) * 0.5 
    img += torch.randn_like(img) * 0.01 
    
    act_buffer = {}
    def hook_fn(m, i, o): act_buffer['val'] = o
    handle = None
    if target_type == 'filter':
        handle = model.features[layer_idx].register_forward_hook(hook_fn)

    for octave in range(octaves):
        img = img.detach().requires_grad_(True)
        optimizer = torch.optim.Adam([img], lr=0.03) # Lower LR for stability
        
        iterations = 100 if target_type == 'class' else 60
        for _ in range(iterations):
            optimizer.zero_grad()
            
            # Stochastic Jitter (forces the model to make features translation-invariant)
            shift_x, shift_y = np.random.randint(-2, 3, 2)
            img_j = torch.roll(img, shifts=(shift_x, shift_y), dims=(2, 3))
            
            out = model(img_j)
            
            # Loss: Target activation
            loss = -out[0, class_idx] if target_type == 'class' else -act_buffer['val'][0, filter_idx].mean()
            
            # 1. TV Loss (Smoothness/Naturalism)
            loss += 0.1 * (torch.abs(img_j[:, :, 1:, :] - img_j[:, :, :-1, :]).mean() + 
                           torch.abs(img_j[:, :, :, 1:] - img_j[:, :, :, :-1]).mean())
            
            # 2. Color Penalty (Prevents oversaturation/neon colors)
            loss += 0.01 * torch.mean((img_j - 0.5)**2)
            
            loss.backward()
            optimizer.step()
            
            with torch.no_grad():
                img.clamp_(0, 1)

        # Upscale & Sharpen for next octave
        if octave < octaves - 1:
            new_size = int(size * scale_factor)
            img = F.interpolate(img, size=(new_size, new_size), mode='bicubic', align_corners=True)
            img = laplacian_sharpen(img, alpha=0.3)
            size = new_size
            
    if handle: handle.remove()
    
    # Final resize to ensure strictly 224x224 output
    img = F.interpolate(img, size=(224, 224), mode='bicubic', align_corners=True)
    return deprocess(img)

# --- INDIVIDUAL EXPORT LOOP ---

CLASSES_TO_VISUALIZE = {
    985: "Daisy_Flower",
}

OUTPUT_BASE = "natural_visualizations"
os.makedirs(OUTPUT_BASE, exist_ok=True)

for class_idx, class_name in CLASSES_TO_VISUALIZE.items():
    print(f"\n>>> PROCESSING CLASS: {class_name}")
    class_path = os.path.join(OUTPUT_BASE, class_name)
    os.makedirs(class_path, exist_ok=True)
    
    # 1. Save Class Archetype
    print("   Creating archetype...")
    archetype = visualize_natural_sharp('class', class_idx=class_idx, octaves=4)
    Image.fromarray(archetype).save(os.path.join(class_path, f"00_{class_name}_archetype.png"))
    
    # 2. Save Filter Visualizations
    for block_name, l_idx in BLOCK_MAP.items():
        print(f"   Generating {block_name} filters...")
        layer_path = os.path.join(class_path, block_name)
        os.makedirs(layer_path, exist_ok=True)
        
        # Get top filters for this specific layer
        top_filters = get_top_filters(l_idx, class_idx, num_filters=3)
        
        for i, f_idx in enumerate(top_filters):
            filter_img = visualize_natural_sharp('filter', layer_idx=l_idx, filter_idx=f_idx, octaves=3)
            
            # Export as individual 224x224 file
            filename = f"rank{i+1}_filterID_{f_idx}.png"
            Image.fromarray(filter_img).save(os.path.join(layer_path, filename))

print(f"\nExecution complete. Files saved to: {os.path.abspath(OUTPUT_BASE)}")


>>> PROCESSING CLASS: Daisy_Flower
   Creating archetype...
   Generating Block1_Conv2 filters...
   Generating Block2_Conv2 filters...
   Generating Block3_Conv3 filters...
   Generating Block4_Conv3 filters...
   Generating Block5_Conv3 filters...

Execution complete. Files saved to: c:\Code\Repo-Home\AT24-Djup-Maskininlärning\Lab\natural_visualizations


## Image Input Highres

In [11]:
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as T
from torchvision import models
from PIL import Image
import torch.nn.functional as F
import os

# --- INITIALIZATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).to(DEVICE).eval()

# VGG16 Layer mapping for structured exploration
BLOCK_MAP = {
    "Block1_Conv2": 2,
    "Block2_Conv2": 7,
    "Block3_Conv3": 14,
    "Block4_Conv3": 21,
    "Block5_Conv3": 28
}

# --- CORE UTILITIES ---

def laplacian_sharpen(img, alpha=0.4):
    """Enhances high-frequency edges to prevent blur during upscaling."""
    blur = T.GaussianBlur(kernel_size=3, sigma=1.0)(img)
    return img + alpha * (img - blur)

def deprocess(img):
    """Converts torch tensor to a clean 0-255 numpy array."""
    img = img.detach().cpu().squeeze(0).numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1) # Standardize range for natural colors
    return (img * 255).astype(np.uint8)

def get_top_filters(layer_idx, class_idx, num_filters=3):
    """Finds the specific neurons in a layer that respond most to the chosen class."""
    img = torch.randn(1, 3, 224, 224).to(DEVICE) * 0.05
    img.requires_grad_(True)
    opt = torch.optim.Adam([img], lr=0.1)
    
    # Quick optimization to find class-relevant activation patterns
    for _ in range(15):
        opt.zero_grad()
        loss = -model(img)[0, class_idx]
        loss.backward()
        opt.step()
        
    activations = []
    def hook(m, i, o): activations.append(o.detach())
    handle = model.features[layer_idx].register_forward_hook(hook)
    model(img)
    handle.remove()
    
    filter_scores = activations[0][0].mean(dim=(1, 2))
    return torch.topk(filter_scores, num_filters).indices.cpu().numpy()


def load_and_preprocess_image(img_path, target_size_int):
    """Loads an image and prepares it as a base tensor for optimization."""
    img = Image.open(img_path).convert('RGB')
    
    # Create a tuple (width, height) for PIL and transforms
    target_size = (target_size_int, target_size_int)
    
    preprocess = T.Compose([
        T.Resize(target_size),
        T.ToTensor(),
        # We REMOVE T.Normalize here so the image stays in [0, 1] range.
        # This allows our clamp_(0, 1) and deprocess() functions to work.
    ])
    
    # Load and move to the correct device (CPU/GPU)
    return preprocess(img).unsqueeze(0).to(DEVICE)

# --- THE NATURAL SHARP ENGINE ---
def load_and_preprocess_image(img_path, target_size_int):
    """Loads an image and prepares it as a base tensor for optimization."""
    img = Image.open(img_path).convert('RGB')
    
    # Create a tuple (width, height) for PIL and transforms
    target_size = (target_size_int, target_size_int)
    
    preprocess = T.Compose([
        T.Resize(target_size),
        T.ToTensor(),
        # We REMOVE T.Normalize here so the image stays in [0, 1] range.
        # This allows our clamp_(0, 1) and deprocess() functions to work.
    ])
    
    # Load and move to the correct device (CPU/GPU)
    return preprocess(img).unsqueeze(0).to(DEVICE)

# --- UPDATED ENGINE ---

def visualize_natural_sharp(target_type, layer_idx=None, filter_idx=None, class_idx=None, octaves=4, scale_factor=1.3):
    # Calculate starting size based on octaves
    size = int(224 / (scale_factor ** (octaves - 1)))
    
    # Load the butterfly image as the starting seed
    # It will be automatically resized to 'size'
    img = load_and_preprocess_image("data/streamlit/daisy_985.jpg", size)

    act_buffer = {}
    def hook_fn(m, i, o): act_buffer['val'] = o
    handle = None
    if target_type == 'filter':
        handle = model.features[layer_idx].register_forward_hook(hook_fn)

    # ImageNet normalization values (moved to device)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

    for octave in range(octaves):
        img = img.detach().requires_grad_(True)
        optimizer = torch.optim.Adam([img], lr=0.01) # Lower LR for image-based starts
        
        iterations = 80 if target_type == 'class' else 50
        for _ in range(iterations):
            optimizer.zero_grad()
            
            # 1. Jitter
            shift_x, shift_y = np.random.randint(-2, 3, 2)
            img_j = torch.roll(img, shifts=(shift_x, shift_y), dims=(2, 3))
            
            # 2. Normalize ONLY for the forward pass
            # This keeps 'img' in [0, 1] range but satisfies the model
            img_normalized = (img_j - mean) / std
            
            out = model(img_normalized)
            
            # 3. Loss
            loss = -out[0, class_idx] if target_type == 'class' else -act_buffer['val'][0, filter_idx].mean()
            
            # Regularization
            loss += 0.1 * (torch.abs(img_j[:, :, 1:, :] - img_j[:, :, :-1, :]).mean() + 
                           torch.abs(img_j[:, :, :, 1:] - img_j[:, :, :, :-1]).mean())
            
            loss.backward()
            optimizer.step()
            
            with torch.no_grad():
                img.clamp_(0, 1)

        # Upscale
        if octave < octaves - 1:
            new_size = int(size * scale_factor)
            img = F.interpolate(img, size=(new_size, new_size), mode='bicubic', align_corners=True)
            img = laplacian_sharpen(img, alpha=0.3)
            size = new_size
            
    if handle: handle.remove()
    return deprocess(img)

# --- INDIVIDUAL EXPORT LOOP ---

CLASSES_TO_VISUALIZE = {
    985: "Daisy_Flower",
}

OUTPUT_BASE = "natural_visualizations_image"
os.makedirs(OUTPUT_BASE, exist_ok=True)

for class_idx, class_name in CLASSES_TO_VISUALIZE.items():
    print(f"\n>>> PROCESSING CLASS: {class_name}")
    class_path = os.path.join(OUTPUT_BASE, class_name)
    os.makedirs(class_path, exist_ok=True)
    
    # 1. Save Class Archetype
    print("   Creating archetype...")
    archetype = visualize_natural_sharp('class', class_idx=class_idx, octaves=4)
    Image.fromarray(archetype).save(os.path.join(class_path, f"00_{class_name}_archetype.png"))
    
    # 2. Save Filter Visualizations
    for block_name, l_idx in BLOCK_MAP.items():
        print(f"   Generating {block_name} filters...")
        layer_path = os.path.join(class_path, block_name)
        os.makedirs(layer_path, exist_ok=True)
        
        # Get top filters for this specific layer
        top_filters = get_top_filters(l_idx, class_idx, num_filters=3)
        
        for i, f_idx in enumerate(top_filters):
            filter_img = visualize_natural_sharp('filter', layer_idx=l_idx, filter_idx=f_idx, octaves=3)
            
            # Export as individual 224x224 file
            filename = f"rank{i+1}_filterID_{f_idx}.png"
            Image.fromarray(filter_img).save(os.path.join(layer_path, filename))

print(f"\nExecution complete. Files saved to: {os.path.abspath(OUTPUT_BASE)}")

# 28 min


>>> PROCESSING CLASS: Daisy_Flower
   Creating archetype...
   Generating Block1_Conv2 filters...
   Generating Block2_Conv2 filters...
   Generating Block3_Conv3 filters...
   Generating Block4_Conv3 filters...
   Generating Block5_Conv3 filters...

Execution complete. Files saved to: c:\Code\Repo-Home\AT24-Djup-Maskininlärning\Lab\natural_visualizations_image


## Noise Generator Isloated

In [28]:
import torch
import numpy as np
from PIL import Image
import os
import torch.nn.functional as F

def generate_vibrant_chunky_noise(num_images, size=224, pixel_size=5, output_dir="vibrant_noise"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Calculate the smaller resolution needed for chunky pixels
    # e.g., 224 / 2 = 112
    low_res_size = size // pixel_size

    for i in range(num_images):
        # 1. Generate Raw Noise
        # Using a wider range (std=0.5 instead of 0.1) to get more blacks and brights
        noise = torch.randn(1, 3, low_res_size, low_res_size) * 0.5 + 0.5
        
        # 2. Increase Contrast/Saturation
        # Pushing values away from the center (0.5) toward the edges (0 and 1)
        noise = (noise - 0.5) * 0.5 + 0.4
        noise = torch.clamp(noise, 0, 1)

        # 3. Upscale to chunky 2x2 pixels
        # 'nearest' mode prevents blurring, keeping the squares sharp
        chunky_noise = F.interpolate(noise, size=(size, size), mode='nearest')

        # 4. Convert to Saveable Format
        img_np = chunky_noise.squeeze(0).cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np * 255).astype(np.uint8)

        # 5. Save
        img = Image.fromarray(img_np)
        filename = f"vibrant_chunky_{i+1:03d}.jpg"
        filepath = os.path.join(output_dir, filename)
        img.save(filepath, "JPEG", quality=98)
        
        print(f"Generated chunky noise: {filepath}")

# --- SETTINGS ---
X = 5 # Number of images
generate_vibrant_chunky_noise(num_images=X)

Generated chunky noise: vibrant_noise\vibrant_chunky_001.jpg
Generated chunky noise: vibrant_noise\vibrant_chunky_002.jpg
Generated chunky noise: vibrant_noise\vibrant_chunky_003.jpg
Generated chunky noise: vibrant_noise\vibrant_chunky_004.jpg
Generated chunky noise: vibrant_noise\vibrant_chunky_005.jpg


## Noise and Original Image Mereger

In [ ]:
from PIL import Image, ImageDraw
import os

def merge_diagonally(path_a, path_b, output_path, line_width=6):
    # 1. Load images
    img_a = Image.open(path_a).convert("RGB")
    img_b = Image.open(path_b).convert("RGB")

    # 2. Match sizes (using Image A as the template)
    width, height = img_a.size
    img_b = img_b.resize((width, height), Image.Resampling.LANCZOS)

    # 3. Create the mask for the diagonal split
    # This creates a black/white image used as a template for the cut
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    # Define the polygon for the top-right half
    draw.polygon([(width, 0), (width, 0), (0, height)], fill=255)

    # 4. Composite the two images using the mask
    # Image A stays on one side, Image B on the other
    result = Image.composite(img_a, img_b, mask)

    # 5. Draw the black diagonal separator line
    draw_line = ImageDraw.Draw(result)
    # Line goes from top-left (0,0) to bottom-right (width, height)
    draw_line.line([(0, height), (width, 0)], fill="black", width=line_width)

    # 6. Save result
    result.save(output_path, "JPEG", quality=95)
    print(f"Successfully created: {output_path}")

# --- EXECUTION ---
# Ensure these files exist in your folder!
path_A = "data/streamlit/original/"
path_B = "data/streamlit/noise/"
path_output = "data/streamlit/merged/"
fileList_A = os.listdir(path_A)
fileList_B = os.listdir(path_B)


for file_A, file_B, in zip(fileList_A, fileList_B):
    merge_diagonally(path_A+file_A, path_B+file_B, f"{path_output}{file_A[:-4]}.jpg")




Successfully created: data/streamlit/merged/baseball_noise_429.jpg
Successfully created: data/streamlit/merged/butterfly_noise_323.jpg
Successfully created: data/streamlit/merged/cat_noise_285.jpg
Successfully created: data/streamlit/merged/daisy_noise_985.jpg
Successfully created: data/streamlit/merged/dog_noise_235.jpg
